In [10]:
# create a document 

from langchain_core.documents import Document

In [11]:
doc = Document(
    page_content = "the is the data for creating rag",
    metadata={
        "source": "internet.com",
        "pages": 23,
        "author" : "ayush",
        "date_created": "2025-03-22"
    }
)
doc

Document(metadata={'source': 'internet.com', 'pages': 23, 'author': 'ayush', 'date_created': '2025-03-22'}, page_content='the is the data for creating rag')

In [9]:
# create a txt file
# import os
# os.makedirs("../data/text_file",exist_ok=False)

In [12]:
# add data in txt file
sample_text = {
    "../data/text_file/training.txt" : """
    A program is many things. It is a piece of text typed by a programmer, it is
the directing force that makes the computer do what it does, it is data in the
computer's memory, and, at the same time, it controls the actions performed
on this memory. Analogies that try to compare programs to familiar objects
tend to fall short. A superficially fitting one is to compare a program to a
machine—lots of separate parts tend to be involved, and to make the whole
thing tick, we have to consider the ways in which these parts interconnect and
contribute to the operation of the whole.
A computer is a physical machine that acts as a host for these immaterial
machines. Computers themselves can do only stupidly straightforward things.
The reason they are so useful is that they do these things at an incredibly
high speed. A program can ingeniously combine an enormous number of these
simple actions to do very complicated things.
A program is a building of thought. It is costless to build, it is weightless,
and it grows easily under our typing hands. But as a program grows, so does
its complexity. The skill of programming is the skill of building programs that
don't confuse the programmer. The best programs are those that manage to
do something interesting while still being easy to understand.
    """
}

for filepath, content in sample_text.items():
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(content)


print("sample data data created!")

sample data data created!


In [16]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_file/training.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_file/training.txt'}, page_content="\n    A program is many things. It is a piece of text typed by a programmer, it is\nthe directing force that makes the computer do what it does, it is data in the\ncomputer's memory, and, at the same time, it controls the actions performed\non this memory. Analogies that try to compare programs to familiar objects\ntend to fall short. A superficially fitting one is to compare a program to a\nmachine—lots of separate parts tend to be involved, and to make the whole\nthing tick, we have to consider the ways in which these parts interconnect and\ncontribute to the operation of the whole.\nA computer is a physical machine that acts as a host for these immaterial\nmachines. Computers themselves can do only stupidly straightforward things.\nThe reason they are so useful is that they do these things at an incredibly\nhigh speed. A program can ingeniously combine an enormous number of these\nsimple actions to do ver

In [17]:
#  load data from a file

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls = PyMuPDFLoader)

pdf_document = dir_loader.load()
print(pdf_document)

[Document(metadata={'producer': 'Skia/PDF m151', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36', 'creationdate': '2026-09-06T07:30:39+00:00', 'source': '../data/pdf/training.pdf', 'file_path': '../data/pdf/training.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': 'Comprehensive Go Programming Language Summary', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-09-06T07:30:39+00:00', 'trapped': '', 'modDate': "D:20260906073039+00'00'", 'creationDate': "D:20260906073039+00'00'", 'page': 0}, page_content='Comprehensive Go Programming Language\nSummary\nhttps://gemini.google.com/app/f49dc32686bca7b4\nUser prompt: create a two page summary on golang.\nResponse: Go (Golang) is an open-source, statically typed, compiled programming language designed\nby Google engineers Robert Griesemer, Rob Pike, and Ken Thompson. Officially launched in 2009, Go\nwas created to solve software engineering challenges at scale:

In [18]:
# data chunking

from langchain_text_splitters import RecursiveCharacterTextSplitter;

documents = dir_loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

print(len(chunks))
print(chunks[0])

23
page_content='Comprehensive Go Programming Language
Summary
https://gemini.google.com/app/f49dc32686bca7b4
User prompt: create a two page summary on golang.
Response: Go (Golang) is an open-source, statically typed, compiled programming language designed' metadata={'producer': 'Skia/PDF m151', 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36', 'creationdate': '2026-09-06T07:30:39+00:00', 'source': '../data/pdf/training.pdf', 'file_path': '../data/pdf/training.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': 'Comprehensive Go Programming Language Summary', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-09-06T07:30:39+00:00', 'trapped': '', 'modDate': "D:20260906073039+00'00'", 'creationDate': "D:20260906073039+00'00'", 'page': 0}


In [19]:
# vector embedding and database
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import *
from sklearn.metrics.pairwise import cosine_similarity


In [23]:
class EmbeddingManager:
    def __init__(self, model_name:str = 'all-MiniLm-L6-v2'):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self) :
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded sucessfully, model dimention: {self.model}")
        except Exception as e:
            print(f"error while loading model {self.model_name}: {e}")
            raise
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("model not loaded")
        

        print(f"generating embedding for {len(texts)} texts")
        embedding = self.model.encode(texts, show_progress_bar = True)
        print(f"generated embedding with shape: {embedding.shape}")
        return embedding
    
embedding_manager=EmbeddingManager()
embedding_manager

loading embedding model: all-MiniLm-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1606.95it/s]


model loaded sucessfully, model dimention: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)
